<a href="https://colab.research.google.com/github/LeonardoGuima/Cyber/blob/main/Analise_Phishing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!git clone --depth 1 https://github.com/rf-peixoto/phishing_pot.git
%cd /content/phishing_pot/email
!ls | head -20

Cloning into 'phishing_pot'...
remote: Enumerating objects: 12276, done.
remote: Counting objects: 100% (12276/12276), done.
remote: Compressing objects: 100% (10299/10299), done.
remote: Total 12276 (delta 1973), reused 12185 (delta 1973), pack-reused 0 (from 0)
Receiving objects: 100% (12276/12276), 970.39 MiB | 23.23 MiB/s, done.
Resolving deltas: 100% (1973/1973), done.
Updating files: 100% (12278/12278), done.
/content/phishing_pot/email
analisa_eml.py
phishing_pot
sample-10000.eml
sample-10001.eml
sample-10002.eml
sample-10003.eml
sample-10004.eml
sample-10005.eml
sample-10006.eml
sample-10007.eml
sample-10008.eml
sample-10009.eml
sample-1000.eml
sample-10010.eml
sample-10011.eml
sample-10012.eml
sample-10013.eml
sample-10014.eml
sample-10015.eml
sample-10016.eml


In [18]:
%%writefile analisa_eml.py
import sys, re, hashlib, email
from email import policy

msg = email.message_from_binary_file(open(sys.argv[1], 'rb'), policy=policy.default)

print("== CABECALHOS ==")
for h in ['From', 'Reply-To', 'Return-Path', 'To', 'Subject', 'Date', 'Message-ID', 'X-Originating-IP']:
    print(f"{h}: {msg.get(h)}")

print("\n== RECEIVED (do primeiro salto ao ultimo) ==")
for i, r in enumerate(reversed(msg.get_all('Received', [])), 1):
    print(f"{i}. {' '.join(str(r).split())[:200]}")

print("\n== AUTENTICACAO ==")
print(msg.get('Authentication-Results'))

print("\n== URLS ==")
urls = set()
for part in msg.walk():
    if part.get_content_maintype() == 'text':
        try:
            urls.update(re.findall(r'https?://[^\s"\'<>]+', part.get_content()))
        except Exception:
            pass
for u in sorted(urls):
    print(u.replace('http', 'hxxp', 1).replace('.', '[.]'))

print("\n== ANEXOS ==")
for part in msg.walk():
    nome = part.get_filename()
    if nome:
        dados = part.get_payload(decode=True) or b''
        print(f"{nome} | {part.get_content_type()} | {len(dados)} bytes | SHA256 {hashlib.sha256(dados).hexdigest()}")


Overwriting analisa_eml.py


In [19]:
%%bash
cd /content/phishing_pot/email
grep -iE '^(From|Reply-To|Return-Path|Subject|Date|Message-ID):' sample-100.eml
grep -i -A2 '^Received:' sample-100.eml
grep -i -A3 '^Authentication-Results:' sample-100.eml
grep -oE 'https?://[^"<> ]+' sample-100.eml | sort -u

Subject: =?UTF-8?B?8J+Uiw==?= Zonnepanelen voor een goede prijs 
From: "Zonnepanelen installateur" <zonnepaneel@appjj.serenitepure.fr>
Reply-To: "Zonnepanelen installateur" <news@aichakandisha.com>
Date: Thu, 3 Nov 2022 04:56:15 +0000
Message-ID: <0.0.0.0.1D8EF409A5C12CE.37AA@dturm.de>
Return-Path: return@dturm.de
Received: from MN2PR19MB3966.namprd19.prod.outlook.com (::1) by
 MN0PR19MB6312.namprd19.prod.outlook.com with HTTPS; Thu, 3 Nov 2022 04:56:17
 +0000
Received: from AM7PR02CA0005.eurprd02.prod.outlook.com (2603:10a6:20b:100::15)
 by MN2PR19MB3966.namprd19.prod.outlook.com (2603:10b6:208:1e8::10) with
 Microsoft SMTP Server (version=TLS1_2,
--
Received: from VI1EUR05FT048.eop-eur05.prod.protection.outlook.com
 (2603:10a6:20b:100::4) by AM7PR02CA0005.outlook.office365.com
 (2603:10a6:20b:100::15) with Microsoft SMTP Server (version=TLS1_2,
--
Received: from dturm.de (57.128.69.202) by
 VI1EUR05FT048.mail.protection.outlook.com (10.233.243.188) with Microsoft
 SMTP Server id 15.2

In [20]:
%%bash
cd /content/phishing_pot/email
grep -iE '^(From|Reply-To|Return-Path|Subject|Date|Message-ID):' sample-200.eml
grep -i -A2 '^Received:' sample-200.eml
grep -i -A3 '^Authentication-Results:' sample-200.eml
grep -oE 'https?://[^"<> ]+' sample-200.eml | sort -u

Subject: Ik denk dat dit wat voor jou is
From: "Mia" <newsmail@appsh.serenitepure.fr>
Reply-To: "Rolanda" <news@aichakandisha.com>
Date: Wed, 28 Dec 2022 19:55:28 +0000
Message-ID: <0.0.0.0.1D91AF65562CED8.2893@bsd-web.de>
Return-Path: returntbmXkbA4@comet-sas.fr
Received: from PH7PR19MB5774.namprd19.prod.outlook.com (::1) by
 MN0PR19MB6312.namprd19.prod.outlook.com with HTTPS; Wed, 28 Dec 2022 19:55:30
 +0000
Received: from FR3P281CA0179.DEUP281.PROD.OUTLOOK.COM (2603:10a6:d10:a0::16)
 by PH7PR19MB5774.namprd19.prod.outlook.com (2603:10b6:510:1d2::6) with
 Microsoft SMTP Server (version=TLS1_2,
--
Received: from VI1EUR04FT021.eop-eur04.prod.protection.outlook.com
 (2603:10a6:d10:a0:cafe::20) by FR3P281CA0179.outlook.office365.com
 (2603:10a6:d10:a0::16) with Microsoft SMTP Server (version=TLS1_2,
--
Received: from bsd-web.de (162.19.55.158) by
 VI1EUR04FT021.mail.protection.outlook.com (10.152.29.63) with Microsoft SMTP
 Server id 15.20.5944.14 via Frontend Transport; Wed, 28 Dec 2022

In [21]:
%%bash
cd /content/phishing_pot/email
grep -iE '^(From|Reply-To|Return-Path|Subject|Date|Message-ID):' sample-300.eml
grep -i -A2 '^Received:' sample-300.eml
grep -i -A3 '^Authentication-Results:' sample-300.eml
grep -oE 'https?://[^"<> ]+' sample-300.eml | sort -u

From: Expédition en attente! , <no-replay@iptesetxkeys.com>
Subject: Répondez au court sondage et recevez votre cadeau.
Date: Wed, 08 Feb 2023 22:39:29 +0100
Message-ID:
Return-Path: service@windows.net
Received: from DM6PR19MB3899.namprd19.prod.outlook.com (::1) by
 MN0PR19MB6312.namprd19.prod.outlook.com with HTTPS; Wed, 8 Feb 2023 22:19:04
 +0000
Received: from BN9PR03CA0968.namprd03.prod.outlook.com (2603:10b6:408:109::13)
 by DM6PR19MB3899.namprd19.prod.outlook.com (2603:10b6:5:22b::20) with
 Microsoft SMTP Server (version=TLS1_2,
--
Received: from BN1NAM02FT014.eop-nam02.prod.protection.outlook.com
 (2603:10b6:408:109:cafe::35) by BN9PR03CA0968.outlook.office365.com
 (2603:10b6:408:109::13) with Microsoft SMTP Server (version=TLS1_2,
--
Received: from iptesetxkeys.com (50.116.26.144) by
 BN1NAM02FT014.mail.protection.outlook.com (10.13.2.131) with Microsoft SMTP
 Server id 15.20.6086.18 via Frontend Transport; Wed, 8 Feb 2023 22:19:03
Authentication-Results: spf=softfail (sender 

In [22]:
!python3 analisa_eml.py sample-100.eml

== CABECALHOS ==
From: Zonnepanelen installateur <zonnepaneel@appjj.serenitepure.fr>
Reply-To: Zonnepanelen installateur <news@aichakandisha.com>
Return-Path: return@dturm.de
To: phishing@pot
Subject: 🔋 Zonnepanelen voor een goede prijs 
Date: Thu, 03 Nov 2022 04:56:15 +0000
Message-ID: <0.0.0.0.1D8EF409A5C12CE.37AA@dturm.de>
X-Originating-IP: None

== RECEIVED (do primeiro salto ao ultimo) ==
1. from dturm.de (57.128.69.202) by VI1EUR05FT048.mail.protection.outlook.com (10.233.243.188) with Microsoft SMTP Server id 15.20.5791.20 via Frontend Transport; Thu, 3 Nov 2022 04:56:15 +0000
2. from VI1EUR05FT048.eop-eur05.prod.protection.outlook.com (2603:10a6:20b:100::4) by AM7PR02CA0005.outlook.office365.com (2603:10a6:20b:100::15) with Microsoft SMTP Server (version=TLS1_2, cipher=TLS_EC
3. from AM7PR02CA0005.eurprd02.prod.outlook.com (2603:10a6:20b:100::15) by MN2PR19MB3966.namprd19.prod.outlook.com (2603:10b6:208:1e8::10) with Microsoft SMTP Server (version=TLS1_2, cipher=TLS_ECDHE_RSA
4

In [23]:
!python3 analisa_eml.py sample-200.eml

== CABECALHOS ==
From: Mia <newsmail@appsh.serenitepure.fr>
Reply-To: Rolanda <news@aichakandisha.com>
Return-Path: returntbmXkbA4@comet-sas.fr
To: phishing@pot
Subject: Ik denk dat dit wat voor jou is
Date: Wed, 28 Dec 2022 19:55:28 +0000
Message-ID: <0.0.0.0.1D91AF65562CED8.2893@bsd-web.de>
X-Originating-IP: None

== RECEIVED (do primeiro salto ao ultimo) ==
1. from bsd-web.de (162.19.55.158) by VI1EUR04FT021.mail.protection.outlook.com (10.152.29.63) with Microsoft SMTP Server id 15.20.5944.14 via Frontend Transport; Wed, 28 Dec 2022 19:55:28 +0000
2. from VI1EUR04FT021.eop-eur04.prod.protection.outlook.com (2603:10a6:d10:a0:cafe::20) by FR3P281CA0179.outlook.office365.com (2603:10a6:d10:a0::16) with Microsoft SMTP Server (version=TLS1_2, cipher=TL
3. from FR3P281CA0179.DEUP281.PROD.OUTLOOK.COM (2603:10a6:d10:a0::16) by PH7PR19MB5774.namprd19.prod.outlook.com (2603:10b6:510:1d2::6) with Microsoft SMTP Server (version=TLS1_2, cipher=TLS_ECDHE_RSA_WI
4. from PH7PR19MB5774.namprd19.pro

In [24]:
!python3 analisa_eml.py sample-300.eml

== CABECALHOS ==
From: "Expédition en attente!", no-replay@iptesetxkeys.com
Reply-To: None
Return-Path: service@windows.net
To: <>
Subject: Répondez au court sondage et recevez votre cadeau.
Date: Wed, 08 Feb 2023 22:39:29 +0100
Message-ID: <ffea458a-8547-4e2d-9158-2f3ff1e1463d@BN1NAM02FT014.eop-nam02.prod.protection.outlook.com>
X-Originating-IP: None

== RECEIVED (do primeiro salto ao ultimo) ==
1. from iptesetxkeys.com (50.116.26.144) by BN1NAM02FT014.mail.protection.outlook.com (10.13.2.131) with Microsoft SMTP Server id 15.20.6086.18 via Frontend Transport; Wed, 8 Feb 2023 22:19:03 +0000
2. from BN1NAM02FT014.eop-nam02.prod.protection.outlook.com (2603:10b6:408:109:cafe::35) by BN9PR03CA0968.outlook.office365.com (2603:10b6:408:109::13) with Microsoft SMTP Server (version=TLS1_2, cipher=
3. from BN9PR03CA0968.namprd03.prod.outlook.com (2603:10b6:408:109::13) by DM6PR19MB3899.namprd19.prod.outlook.com (2603:10b6:5:22b::20) with Microsoft SMTP Server (version=TLS1_2, cipher=TLS_ECDH

In [25]:
!cat sample-100.eml

Received: from MN2PR19MB3966.namprd19.prod.outlook.com (::1) by
 MN0PR19MB6312.namprd19.prod.outlook.com with HTTPS; Thu, 3 Nov 2022 04:56:17
 +0000
Received: from AM7PR02CA0005.eurprd02.prod.outlook.com (2603:10a6:20b:100::15)
 by MN2PR19MB3966.namprd19.prod.outlook.com (2603:10b6:208:1e8::10) with
 Microsoft SMTP Server (version=TLS1_2,
 cipher=TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384) id 15.20.5769.16; Thu, 3 Nov
 2022 04:56:16 +0000
Received: from VI1EUR05FT048.eop-eur05.prod.protection.outlook.com
 (2603:10a6:20b:100::4) by AM7PR02CA0005.outlook.office365.com
 (2603:10a6:20b:100::15) with Microsoft SMTP Server (version=TLS1_2,
 cipher=TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384) id 15.20.5791.22 via Frontend
 Transport; Thu, 3 Nov 2022 04:56:15 +0000
Authentication-Results: spf=none (sender IP is 57.128.69.202)
 smtp.mailfrom=dturm.de; dkim=none (message not signed)
 header.d=none;dmarc=none action=none
 header.from=appjj.serenitepure.fr;compauth=fail reason=001
Received-SPF: None (protect